# 10_SageMaker_MLOps_Pipeline.ipynb
## Heart Attack Risk Assessment — SageMaker MLOps Pipeline

**Pipeline name:** `iti113-team05-heart-attack-risk`  
**Region:** `ap-southeast-1`

### Important update in this version
This notebook **does not create new `preprocess.py`, `train.py`, or `evaluate.py` implementations**.

Instead it:
1. discovers your **existing project scripts**;
2. prints the selected paths;
3. performs a basic SageMaker-runtime contract check;
4. uses those existing scripts in the pipeline;
5. keeps your original script files unchanged.

The pipeline is:

```text
full_train_raw.csv
      ↓
Existing preprocess.py
      ↓
Existing train.py
      ↓
Existing evaluate.py
      ↓
PR-AUC + Recall quality gate
      ↓
PASS → Model Registry (PendingManualApproval)
FAIL → Stop
```

### Governance
The locked Stage 8 holdout is not reused in this repeatable pipeline. Notebook 10 should operate on training/validation data only.

### Relationship to later notebooks
- **Notebook 10:** pipeline orchestration
- **Notebook 11:** controlled endpoint deployment
- **Notebook 12:** Gradio demo

## When do I run Notebook 10?

You do **not** need to run Notebook 10 every time you recreate an endpoint.

```text
New/changed data or model
        ↓
Notebook 10
        ↓
new registered model version
        ↓
Notebook 11
        ↓
endpoint
```

If the endpoint was deleted only to save AWS cost and you want the **same model** again:

```text
Notebook 11 only
```

## 1. FIRST RUN — Install/verify SageMaker SDK v3 packages

In [1]:
%pip install -q -U sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops joblib pandas scikit-learn xgboost
print("Packages installed/verified.")

Note: you may need to restart the kernel to use updated packages.
Packages installed/verified.


## 2. FIRST RUN — Imports and AWS/session setup

In [2]:
import os
import json
import re
import shutil
from pathlib import Path

import boto3
import joblib
import pandas as pd
import sklearn
import xgboost

from sklearn.pipeline import Pipeline
from sklearn.base import clone
from xgboost import XGBClassifier

from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.workflow.pipeline_context import PipelineSession
from sagemaker.core import image_uris
from sagemaker.core.processing import ScriptProcessor
from sagemaker.core.shapes import (
    ProcessingInput, ProcessingS3Input,
    ProcessingOutput, ProcessingS3Output,
)
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import SourceCode, Compute, InputData
from sagemaker.mlops.workflow.pipeline import Pipeline as SageMakerPipeline
from sagemaker.mlops.workflow.steps import ProcessingStep, TrainingStep, CacheConfig
from sagemaker.mlops.workflow.model_step import ModelStep
from sagemaker.mlops.workflow.condition_step import ConditionStep
from sagemaker.core.workflow.parameters import ParameterInteger, ParameterString, ParameterFloat
from sagemaker.core.workflow.properties import PropertyFile
from sagemaker.core.workflow.functions import JsonGet
from sagemaker.core.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.core.model_metrics import MetricsSource, ModelMetrics

REGION = "ap-southeast-1"
os.environ["AWS_DEFAULT_REGION"] = REGION
os.environ["AWS_REGION"] = REGION
boto3.setup_default_session(region_name=REGION)

boto_session = boto3.Session(region_name=REGION)
sm_session = Session(boto_session=boto_session)
pipeline_session = PipelineSession(boto_session=boto_session)

ROLE_ARN = get_execution_role()
BUCKET = sm_session.default_bucket()

sm_client = boto3.client("sagemaker", region_name=REGION)
s3_client = boto3.client("s3", region_name=REGION)

print("Region :", REGION)
print("Role   :", ROLE_ARN)
print("Bucket :", BUCKET)
print("sklearn:", sklearn.__version__)
print("xgboost:", xgboost.__version__)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Region : ap-southeast-1
Role   : arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team05
Bucket : sagemaker-ap-southeast-1-044528205969
sklearn: 1.9.0
xgboost: 3.4.1


## 3. FIRST RUN — Locate project artifacts

In [3]:
EXPECTED_PROJECT_FOLDER = "Heart_Attack_Risk_Assessment"
current = Path.cwd().resolve()

if current.name == EXPECTED_PROJECT_FOLDER:
    PROJECT_ROOT = current
else:
    PROJECT_ROOT = next((p for p in current.parents if p.name == EXPECTED_PROJECT_FOLDER), None)

if PROJECT_ROOT is None:
    fallback = Path("/home/sagemaker-user/Heart_Attack_Risk_Assessment")
    if fallback.exists():
        PROJECT_ROOT = fallback
    else:
        raise FileNotFoundError("Cannot locate Heart_Attack_Risk_Assessment.")

DATA_DIR = PROJECT_ROOT / "data"
CONFIG_DIR = PROJECT_ROOT / "config"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
STAGE6_DIR = ARTIFACT_DIR / "stage6"

PIPELINE_WORK_DIR = ARTIFACT_DIR / "stage10_pipeline"
PIPELINE_WORK_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_RAW_FILE = DATA_DIR / "full_train_raw.csv"
FROZEN_MODEL_FILE = STAGE6_DIR / "models" / "xgboost_best_estimator.joblib"
STAGE6_NOMINATION_FILE = CONFIG_DIR / "stage6_candidate_nomination.json"
FEATURE_METADATA_FILE = CONFIG_DIR / "feature_metadata.json"

required = {
    "full training data": TRAIN_RAW_FILE,
    "Stage 6 XGBoost pipeline": FROZEN_MODEL_FILE,
    "Stage 6 nomination": STAGE6_NOMINATION_FILE,
    "feature metadata": FEATURE_METADATA_FILE,
}

missing=[]
for name,path in required.items():
    print(f"{'FOUND' if path.exists() else 'MISSING':7} | {name:28} | {path}")
    if not path.exists():
        missing.append(str(path))

if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

print("\n[OK] Core project artifacts found.")

FOUND   | full training data           | /home/sagemaker-user/Heart_Attack_Risk_Assessment/data/full_train_raw.csv
FOUND   | Stage 6 XGBoost pipeline     | /home/sagemaker-user/Heart_Attack_Risk_Assessment/artifacts/stage6/models/xgboost_best_estimator.joblib
FOUND   | Stage 6 nomination           | /home/sagemaker-user/Heart_Attack_Risk_Assessment/config/stage6_candidate_nomination.json
FOUND   | feature metadata             | /home/sagemaker-user/Heart_Attack_Risk_Assessment/config/feature_metadata.json

[OK] Core project artifacts found.


## 4. FIRST RUN — Extract configuration from legacy Stage 6 model

The legacy joblib is loaded **once only to recover configuration**.  
The fitted XGBoost Booster/tree state is not saved into the new training template.

In [4]:
import warnings

with open(STAGE6_NOMINATION_FILE, "r", encoding="utf-8") as f:
    stage6_nomination = json.load(f)

with open(FEATURE_METADATA_FILE, "r", encoding="utf-8") as f:
    feature_metadata = json.load(f)

FROZEN_THRESHOLD = float(stage6_nomination["selected_threshold"])
TARGET = "HadHeartAttack"
FULL_FEATURES = feature_metadata["full"]["features"]

print("=" * 80)
print("SECTION 4 — LEGACY CONFIGURATION EXTRACTION")
print("=" * 80)
print("Current XGBoost version :", xgboost.__version__)
print("Current sklearn version :", sklearn.__version__)
print("Candidate               :", stage6_nomination["leading_candidate"])
print("Frozen threshold        :", FROZEN_THRESHOLD)
print("Feature count           :", len(FULL_FEATURES))

with warnings.catch_warnings():
    warnings.simplefilter("default")
    legacy_pipeline = joblib.load(FROZEN_MODEL_FILE)

if not isinstance(legacy_pipeline, Pipeline):
    raise TypeError("Stage 6 artifact is not an sklearn Pipeline.")

PIPELINE_STEP_NAMES = list(legacy_pipeline.named_steps.keys())

print("\nPipeline steps:")
for i,(name,obj) in enumerate(legacy_pipeline.steps, 1):
    print(f"{i}. {name:<25} {type(obj).__name__}")

xgb_step_name = None
legacy_xgb = None
for name,obj in legacy_pipeline.steps:
    if isinstance(obj, XGBClassifier):
        xgb_step_name = name
        legacy_xgb = obj
        break

if legacy_xgb is None:
    raise RuntimeError("Could not locate XGBClassifier in Stage 6 pipeline.")

XGB_HYPERPARAMETERS = legacy_xgb.get_params(deep=False)

PREPROCESSING_STEPS = []
for name,obj in legacy_pipeline.steps:
    if name == xgb_step_name:
        break
    PREPROCESSING_STEPS.append((name, clone(obj)))

print("\nXGBoost step            :", xgb_step_name)
print("Preprocessing steps     :", [n for n,_ in PREPROCESSING_STEPS])

legacy_xgb = None
legacy_pipeline = None

print("\n✅ Configuration extracted.")
print("✅ Legacy fitted XGBoost state will not be reused.")

SECTION 4 — LEGACY CONFIGURATION EXTRACTION
Current XGBoost version : 3.4.1
Current sklearn version : 1.9.0
Candidate               : xgboost
Frozen threshold        : 0.52
Feature count           : 39



Pipeline steps:
1. preprocessor              ColumnTransformer
2. model                     XGBClassifier

XGBoost step            : model
Preprocessing steps     : ['preprocessor']

✅ Configuration extracted.
✅ Legacy fitted XGBoost state will not be reused.


/opt/conda/lib/python3.12/pickle.py:1760: UserWarning: [13:07:58] WARNING: /__w/xgboost/xgboost/src/collective/../data/../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


## 5. FIRST RUN — Build clean UNFITTED training template

In [5]:
clean_preprocessing_steps = [
    (name, clone(obj))
    for name,obj in PREPROCESSING_STEPS
]

fresh_xgb = XGBClassifier(**XGB_HYPERPARAMETERS)

clean_training_pipeline = Pipeline(
    clean_preprocessing_steps + [(xgb_step_name, fresh_xgb)]
)

try:
    fresh_xgb.get_booster()
    raise RuntimeError("Unexpected fitted XGBoost state detected.")
except RuntimeError as e:
    if "Unexpected fitted" in str(e):
        raise
    print("✅ Fresh XGBoost estimator is unfitted.")
except Exception:
    print("✅ Fresh XGBoost estimator is unfitted.")

TEMPLATE_PIPELINE_FILE = PIPELINE_WORK_DIR / "template_pipeline.joblib"
METADATA_TEMPLATE_FILE = PIPELINE_WORK_DIR / "metadata_template.json"

joblib.dump(clean_training_pipeline, TEMPLATE_PIPELINE_FILE)

metadata_template = {
    "project": "Heart_Attack_Risk_Assessment",
    "model": "xgboost",
    "threshold": FROZEN_THRESHOLD,
    "features": FULL_FEATURES,
    "target": TARGET,
    "legacy_fitted_state_reused": False,
    "future_event_prediction": False,
    "diagnostic_tool": False,
    "disclaimer": (
        "Educational prototype only. This output describes similarity to "
        "respondents who reported a previous heart attack. It does not predict "
        "a future heart attack and is not a medical diagnosis."
    ),
}
METADATA_TEMPLATE_FILE.write_text(json.dumps(metadata_template, indent=2), encoding="utf-8")

print("\nCurrent XGBoost version :", xgboost.__version__)
print("Candidate               :", stage6_nomination["leading_candidate"])
print("Feature count           :", len(FULL_FEATURES))
print("Pipeline steps          :", list(clean_training_pipeline.named_steps.keys()))
print("Legacy fitted state     : NOT REUSED")
print("Training template       : UNFITTED")
print("\nSAFE TO CONTINUE.")

✅ Fresh XGBoost estimator is unfitted.

Current XGBoost version : 3.4.1
Candidate               : xgboost
Feature count           : 39
Pipeline steps          : ['preprocessor', 'model']
Legacy fitted state     : NOT REUSED
Training template       : UNFITTED

SAFE TO CONTINUE.


## 6. FIRST RUN — Discover EXISTING project scripts

This section does **not** write new preprocessing/training/evaluation implementations.

It searches the project for:
- `preprocess.py`
- `train.py`
- `evaluate.py`

and ignores the previous generated `src/pipeline10` directory if it exists.

If there are multiple candidates, the notebook prints them. You can explicitly override the selected paths in this cell.

In [6]:
# Optional explicit overrides.
# Leave as None to auto-discover.
PREPROCESS_SCRIPT_OVERRIDE = None
TRAIN_SCRIPT_OVERRIDE = None
EVALUATE_SCRIPT_OVERRIDE = None

EXCLUDED_PARTS = {
    "pipeline10",
    "stage10_pipeline",
    ".ipynb_checkpoints",
}

def candidates_for(filename):
    matches=[]
    for p in PROJECT_ROOT.rglob(filename):
        if not p.is_file():
            continue
        rel_parts=set(p.relative_to(PROJECT_ROOT).parts)
        if rel_parts & EXCLUDED_PARTS:
            continue
        matches.append(p.resolve())
    return sorted(matches, key=lambda p: (len(p.parts), str(p)))

def choose_script(filename, override):
    if override:
        p=Path(override).resolve()
        if not p.exists():
            raise FileNotFoundError(f"Override does not exist: {p}")
        return p

    matches=candidates_for(filename)
    print(f"\nCandidates for {filename}:")
    if not matches:
        print("  NONE")
        return None

    for i,p in enumerate(matches,1):
        print(f"  {i}. {p}")

    # Prefer src/<filename>, then scripts/<filename>, otherwise shortest path.
    preferred = [
        PROJECT_ROOT / "src" / filename,
        PROJECT_ROOT / "scripts" / filename,
    ]
    for p in preferred:
        if p.resolve() in matches:
            return p.resolve()

    return matches[0]

PREPROCESS_SCRIPT = choose_script("preprocess.py", PREPROCESS_SCRIPT_OVERRIDE)
TRAIN_SCRIPT = choose_script("train.py", TRAIN_SCRIPT_OVERRIDE)
EVALUATE_SCRIPT = choose_script("evaluate.py", EVALUATE_SCRIPT_OVERRIDE)

selected = {
    "preprocess.py": PREPROCESS_SCRIPT,
    "train.py": TRAIN_SCRIPT,
    "evaluate.py": EVALUATE_SCRIPT,
}

print("\n" + "="*80)
print("SELECTED EXISTING SCRIPTS")
print("="*80)

missing_scripts=[]
for name,p in selected.items():
    print(f"{name:<15}: {p}")
    if p is None:
        missing_scripts.append(name)

if missing_scripts:
    raise FileNotFoundError(
        "Could not find existing script(s): " + ", ".join(missing_scripts) +
        "\nSet the *_SCRIPT_OVERRIDE values above if your files use different paths."
    )

print("\n✅ Existing project scripts selected.")
print("✅ No duplicate ML implementation has been generated.")


Candidates for preprocess.py:
  1. /home/sagemaker-user/Heart_Attack_Risk_Assessment/src/preprocess.py

Candidates for train.py:
  1. /home/sagemaker-user/Heart_Attack_Risk_Assessment/src/train.py

Candidates for evaluate.py:
  1. /home/sagemaker-user/Heart_Attack_Risk_Assessment/src/evaluate.py

SELECTED EXISTING SCRIPTS
preprocess.py  : /home/sagemaker-user/Heart_Attack_Risk_Assessment/src/preprocess.py
train.py       : /home/sagemaker-user/Heart_Attack_Risk_Assessment/src/train.py
evaluate.py    : /home/sagemaker-user/Heart_Attack_Risk_Assessment/src/evaluate.py

✅ Existing project scripts selected.
✅ No duplicate ML implementation has been generated.


## 7. FIRST RUN — Basic SageMaker contract check for existing scripts

This check does **not** change the scripts. It only warns if the selected source does not appear to contain the runtime paths typically used by SageMaker jobs.

If your existing scripts use command-line arguments rather than these exact environment paths, a warning does not automatically mean the script is wrong.

In [7]:
def read_text(path):
    return Path(path).read_text(encoding="utf-8", errors="ignore")

pre_text = read_text(PREPROCESS_SCRIPT)
train_text = read_text(TRAIN_SCRIPT)
eval_text = read_text(EVALUATE_SCRIPT)

checks = {
    "preprocess.py": [
        ("/opt/ml/processing", "/opt/ml/processing"),
    ],
    "train.py": [
        ("SM_MODEL_DIR or /opt/ml/model", r"SM_MODEL_DIR|/opt/ml/model"),
        ("training channel", r"SM_CHANNEL|/opt/ml/input/data"),
    ],
    "evaluate.py": [
        ("/opt/ml/processing", "/opt/ml/processing"),
        ("evaluation.json", "evaluation.json"),
    ],
}

texts = {
    "preprocess.py": pre_text,
    "train.py": train_text,
    "evaluate.py": eval_text,
}

for name,expected in checks.items():
    print(f"\n{name}")
    print("-"*60)
    for label,pattern in expected:
        ok = re.search(pattern, texts[name], flags=re.I) is not None
        print(f"{'OK' if ok else 'WARN':5} | {label}")

print(
    "\nNOTE: WARN means review the script interface before executing the pipeline. "
    "The notebook intentionally does not rewrite your existing ML scripts."
)


preprocess.py
------------------------------------------------------------
OK    | /opt/ml/processing

train.py
------------------------------------------------------------
OK    | SM_MODEL_DIR or /opt/ml/model
WARN  | training channel

evaluate.py
------------------------------------------------------------
OK    | /opt/ml/processing
OK    | evaluation.json

NOTE: WARN means review the script interface before executing the pipeline. The notebook intentionally does not rewrite your existing ML scripts.


## 8. FIRST RUN — Stage copies of existing scripts for SageMaker packaging

The original files remain unchanged. This creates a **deployment source bundle** containing copies of your existing scripts plus the clean template/metadata/requirements needed by managed jobs.

This is packaging only; it does not create new ML logic.

In [8]:
PIPELINE_SOURCE_DIR = PIPELINE_WORK_DIR / "source_bundle"
if PIPELINE_SOURCE_DIR.exists():
    shutil.rmtree(PIPELINE_SOURCE_DIR)
PIPELINE_SOURCE_DIR.mkdir(parents=True, exist_ok=True)

staged_preprocess = PIPELINE_SOURCE_DIR / "preprocess.py"
staged_train = PIPELINE_SOURCE_DIR / "train.py"
staged_evaluate = PIPELINE_SOURCE_DIR / "evaluate.py"

shutil.copy2(PREPROCESS_SCRIPT, staged_preprocess)
shutil.copy2(TRAIN_SCRIPT, staged_train)
shutil.copy2(EVALUATE_SCRIPT, staged_evaluate)
shutil.copy2(TEMPLATE_PIPELINE_FILE, PIPELINE_SOURCE_DIR / "template_pipeline.joblib")
shutil.copy2(METADATA_TEMPLATE_FILE, PIPELINE_SOURCE_DIR / "metadata_template.json")

REQUIREMENTS_FILE = PIPELINE_SOURCE_DIR / "requirements.txt"
REQUIREMENTS_FILE.write_text(
    f"numpy\npandas\njoblib\nscikit-learn=={sklearn.__version__}\nxgboost=={xgboost.__version__}\n",
    encoding="utf-8"
)

print("Staged source bundle:")
for p in sorted(PIPELINE_SOURCE_DIR.iterdir()):
    print(" -", p.name)

print("\n✅ Existing scripts copied for job packaging.")
print("✅ Original source files were not changed.")

Staged source bundle:
 - evaluate.py
 - metadata_template.json
 - preprocess.py
 - requirements.txt
 - template_pipeline.joblib
 - train.py

✅ Existing scripts copied for job packaging.
✅ Original source files were not changed.


## 9. FIRST RUN — Upload training data to S3

In [9]:
PIPELINE_NAME = "iti113-team05-heart-attack-risk"
MODEL_PACKAGE_GROUP = "iti113-team05-heart-attack-risk-models"
S3_PREFIX = "heart-attack-risk/pipeline10"

RAW_INPUT_KEY = f"{S3_PREFIX}/input/full_train_raw.csv"
RAW_INPUT_S3_URI = f"s3://{BUCKET}/{RAW_INPUT_KEY}"

s3_client.upload_file(str(TRAIN_RAW_FILE), BUCKET, RAW_INPUT_KEY)

print("Pipeline name       :", PIPELINE_NAME)
print("Model package group :", MODEL_PACKAGE_GROUP)
print("Input data          :", RAW_INPUT_S3_URI)

Pipeline name       : iti113-team05-heart-attack-risk
Model package group : iti113-team05-heart-attack-risk-models
Input data          : s3://sagemaker-ap-southeast-1-044528205969/heart-attack-risk/pipeline10/input/full_train_raw.csv


## 10. FIRST RUN — Pipeline parameters and quality gates

In [10]:
processing_instance_count = ParameterInteger("ProcessingInstanceCount", default_value=1)
processing_instance_type = ParameterString("ProcessingInstanceType", default_value="ml.m5.large")
training_instance_type = ParameterString("TrainingInstanceType", default_value="ml.m5.large")
input_data = ParameterString("InputDataUrl", default_value=RAW_INPUT_S3_URI)
model_approval_status = ParameterString("ModelApprovalStatus", default_value="PendingManualApproval")
min_pr_auc = ParameterFloat("MinimumPRAUC", default_value=0.35)
min_recall = ParameterFloat("MinimumRecall", default_value=0.75)

cache_config = CacheConfig(enable_caching=True, expire_after="30d")

print("PR-AUC gate        :", 0.35)
print("Recall gate        :", 0.75)
print("Registration status: PendingManualApproval")

PR-AUC gate        : 0.35
Recall gate        : 0.75
Registration status: PendingManualApproval


## 11. FIRST RUN — Resolve SageMaker Scikit-learn image

In [11]:
SKLEARN_CONTAINER_VERSION = "1.4-2-py312"

SKLEARN_IMAGE_URI = image_uris.retrieve(
    framework="sklearn",
    region=REGION,
    version=SKLEARN_CONTAINER_VERSION,
    image_scope="training",
    instance_type="ml.m5.large",
)

print(SKLEARN_IMAGE_URI)

[08/15/26 13:08:36] INFO     Defaulting to only available Python version: py3                     ]8;id=8816999;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=8817000;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#615\615]8;;\

121021644041.dkr.ecr.ap-southeast-1.amazonaws.com/sagemaker-scikit-learn:1.4-2-py312-cpu-py3


## 12. FIRST RUN — Define preprocessing step using EXISTING `preprocess.py`

**Important:** This assumes your existing preprocessing script can run as a SageMaker Processing job and write:
- `/opt/ml/processing/train`
- `/opt/ml/processing/validation`

If Section 7 showed warnings, inspect the existing script before starting the pipeline.

In [12]:
# ============================================================
# SECTION 12A — CREATE PIPELINE SPLIT SCRIPT
# full_train_raw.csv is ALREADY preprocessed/encoded
# ============================================================

PIPELINE_SPLIT_SCRIPT = (
    PIPELINE_WORK_DIR /
    "pipeline_split.py"
)

PIPELINE_SPLIT_SCRIPT.write_text(
r'''
import argparse
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split


TARGET = "HadHeartAttack"


def main():

    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--input-csv",
        required=True
    )

    parser.add_argument(
        "--output-dir",
        default="/opt/ml/processing/output"
    )

    parser.add_argument(
        "--validation-size",
        type=float,
        default=0.20
    )

    parser.add_argument(
        "--random-state",
        type=int,
        default=42
    )

    args = parser.parse_args()

    input_path = Path(
        args.input_csv
    )

    output_dir = Path(
        args.output_dir
    )

    train_dir = (
        output_dir / "train"
    )

    validation_dir = (
        output_dir / "validation"
    )

    train_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    validation_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    # --------------------------------------------------------
    # Load ALREADY-PREPROCESSED Stage 3 training data
    # --------------------------------------------------------

    df = pd.read_csv(
        input_path
    )

    print(
        "Input shape:",
        df.shape
    )

    print(
        "Target values:",
        sorted(
            df[TARGET]
            .dropna()
            .unique()
            .tolist()
        )
    )


    if TARGET not in df.columns:

        raise KeyError(
            f"Missing target: {TARGET}"
        )


    # Target should already be numeric 0/1
    valid_values = set(
        df[TARGET]
        .dropna()
        .unique()
        .tolist()
    )

    if not valid_values.issubset(
        {0, 1}
    ):

        raise ValueError(
            "Expected HadHeartAttack "
            "to already be encoded as 0/1. "
            f"Found: {valid_values}"
        )


    # --------------------------------------------------------
    # Internal train/validation split
    # Locked Stage 8 holdout is NOT involved.
    # --------------------------------------------------------

    train_df, validation_df = (
        train_test_split(
            df,
            test_size=
                args.validation_size,
            random_state=
                args.random_state,
            stratify=
                df[TARGET],
        )
    )


    train_df.to_csv(
        train_dir / "train.csv",
        index=False
    )

    validation_df.to_csv(
        validation_dir /
        "validation.csv",
        index=False
    )


    print(
        "Train shape:",
        train_df.shape
    )

    print(
        "Validation shape:",
        validation_df.shape
    )

    print(
        "Train prevalence:",
        float(
            train_df[TARGET].mean()
        )
    )

    print(
        "Validation prevalence:",
        float(
            validation_df[TARGET].mean()
        )
    )

    print(
        "Locked holdout used: False"
    )


if __name__ == "__main__":
    main()
''',
    encoding="utf-8"
)

print(
    "Created:",
    PIPELINE_SPLIT_SCRIPT
)

print(
    "Exists:",
    PIPELINE_SPLIT_SCRIPT.exists()
)

Created: /home/sagemaker-user/Heart_Attack_Risk_Assessment/artifacts/stage10_pipeline/pipeline_split.py
Exists: True


In [13]:
# ============================================================
# SECTION 12B — DEFINE PIPELINE SPLIT PROCESSING STEP
# ============================================================

processor = ScriptProcessor(
    image_uri=SKLEARN_IMAGE_URI,
    role=ROLE_ARN,
    instance_count=
        processing_instance_count,
    instance_type=
        processing_instance_type,
    command=["python3"],
    base_job_name=
        "team05-heart-split",
    sagemaker_session=
        pipeline_session,
)


process_args = processor.run(

    code=str(
        PIPELINE_SPLIT_SCRIPT
    ),

    inputs=[
        ProcessingInput(
            input_name="raw-train",

            s3_input=
                ProcessingS3Input(

                    s3_uri=
                        input_data,

                    local_path=
                        "/opt/ml/processing/input",

                    s3_data_type=
                        "S3Prefix",

                    s3_input_mode=
                        "File",

                    s3_data_distribution_type=
                        "FullyReplicated",
                ),
        )
    ],

    outputs=[

        ProcessingOutput(
            output_name="train",

            s3_output=
                ProcessingS3Output(

                    s3_uri=
                        f"s3://{BUCKET}/"
                        f"{S3_PREFIX}/train",

                    local_path=
                        "/opt/ml/processing/"
                        "output/train",

                    s3_upload_mode=
                        "EndOfJob",
                ),
        ),

        ProcessingOutput(
            output_name="validation",

            s3_output=
                ProcessingS3Output(

                    s3_uri=
                        f"s3://{BUCKET}/"
                        f"{S3_PREFIX}/validation",

                    local_path=
                        "/opt/ml/processing/"
                        "output/validation",

                    s3_upload_mode=
                        "EndOfJob",
                ),
        ),
    ],

    arguments=[
        "--input-csv",
        "/opt/ml/processing/input/"
        "full_train_raw.csv",

        "--output-dir",
        "/opt/ml/processing/output",

        "--validation-size",
        "0.20",

        "--random-state",
        "42",
    ],
)


step_process = ProcessingStep(

    name=
        "PreprocessHeartAttackData",

    step_args=
        process_args,

    cache_config=
        cache_config,
)


print(
    "[OK] Pipeline split step defined."
)

print(
    "Input: already-preprocessed "
    "full_train_raw.csv"
)

print(
    "Locked holdout reused: NO"
)

[08/15/26 13:08:53] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=8817007;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=8817008;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#304\304]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

/opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


[OK] Pipeline split step defined.
Input: already-preprocessed full_train_raw.csv
Locked holdout reused: NO


## 13. FIRST RUN — Define training step using EXISTING `train.py`

The complete staged source bundle is supplied to the training job so that local imports and the clean configuration template can travel with the job.

Your existing `train.py` remains the training implementation.

In [14]:
# ============================================================
# SECTION 12C — PREPARE TRAINING SUPPORT FILES
# ============================================================

import shutil

FEATURE_METADATA_SOURCE = (
    PROJECT_ROOT
    / "config"
    / "feature_metadata.json"
)

FEATURE_METADATA_STAGED = (
    PIPELINE_SOURCE_DIR
    / "feature_metadata.json"
)

if not FEATURE_METADATA_SOURCE.exists():
    raise FileNotFoundError(
        f"Feature metadata not found: "
        f"{FEATURE_METADATA_SOURCE}"
    )

shutil.copy2(
    FEATURE_METADATA_SOURCE,
    FEATURE_METADATA_STAGED
)

print("Feature metadata source:")
print(FEATURE_METADATA_SOURCE)

print("\nStaged as:")
print(FEATURE_METADATA_STAGED)

print("\nExists:")
print(FEATURE_METADATA_STAGED.exists())

print("\n✅ feature_metadata.json staged for training.")

Feature metadata source:
/home/sagemaker-user/Heart_Attack_Risk_Assessment/config/feature_metadata.json

Staged as:
/home/sagemaker-user/Heart_Attack_Risk_Assessment/artifacts/stage10_pipeline/source_bundle/feature_metadata.json

Exists:
True

✅ feature_metadata.json staged for training.


In [15]:
# ============================================================
# SECTION 12D — FIX TRAINING REQUIREMENTS
# ============================================================

REQUIREMENTS_FILE = (
    PIPELINE_SOURCE_DIR
    / "requirements.txt"
)

REQUIREMENTS_FILE.write_text(
"""numpy
pandas
joblib
xgboost==3.4.1
""",
    encoding="utf-8"
)

print("Training requirements:")
print(REQUIREMENTS_FILE.read_text())

print(
    "✅ SageMaker container's built-in "
    "scikit-learn 1.4.2 will be retained."
)

Training requirements:
numpy
pandas
joblib
xgboost==3.4.1

✅ SageMaker container's built-in scikit-learn 1.4.2 will be retained.


In [16]:
# ============================================================
# SECTION 13 — DEFINE TRAINING STEP
# Existing train.py + required command-line arguments
# ============================================================

from sagemaker.train import ModelTrainer
from sagemaker.train.configs import (
    SourceCode,
    Compute,
    InputData,
)


print("=" * 80)
print("SECTION 13 — TRAINING STEP")
print("=" * 80)


# ------------------------------------------------------------
# 1. Source code
# ------------------------------------------------------------

training_source = SourceCode(
    source_dir=str(
        PIPELINE_SOURCE_DIR
    ),
    entry_script="train.py",
    requirements="requirements.txt",
)


# ------------------------------------------------------------
# 2. Paths INSIDE SageMaker training container
# ------------------------------------------------------------

TRAIN_CSV_CONTAINER = (
    "/opt/ml/input/data/train/"
    "train.csv"
)

FEATURE_METADATA_CONTAINER = (
    "/opt/ml/input/data/code/"
    "feature_metadata.json"
)

MODEL_OUTPUT_DIR = (
    "/opt/ml/model"
)


# ------------------------------------------------------------
# 3. Hyperparameters
#
# SageMaker passes these to train.py as CLI arguments.
#
# Example:
#   train-csv
# becomes:
#   --train-csv
# ------------------------------------------------------------

training_hyperparameters = {

    "train-csv":
        TRAIN_CSV_CONTAINER,

    "feature-metadata":
        FEATURE_METADATA_CONTAINER,

    "feature-set":
        "full",

    "model":
        "xgboost",

    "output-dir":
        MODEL_OUTPUT_DIR,

    "random-state":
        42,

    "xgb-n-estimators":
        int(
            XGB_HYPERPARAMETERS[
                "n_estimators"
            ]
        ),

    "xgb-learning-rate":
        float(
            XGB_HYPERPARAMETERS[
                "learning_rate"
            ]
        ),

    "xgb-max-depth":
        int(
            XGB_HYPERPARAMETERS[
                "max_depth"
            ]
        ),

    "xgb-subsample":
        float(
            XGB_HYPERPARAMETERS[
                "subsample"
            ]
        ),

    "xgb-colsample-bytree":
        float(
            XGB_HYPERPARAMETERS[
                "colsample_bytree"
            ]
        ),

    "xgb-reg-alpha":
        float(
            XGB_HYPERPARAMETERS[
                "reg_alpha"
            ]
        ),

    "xgb-reg-lambda":
        float(
            XGB_HYPERPARAMETERS[
                "reg_lambda"
            ]
        ),

    "xgb-scale-pos-weight":
        float(
            XGB_HYPERPARAMETERS[
                "scale_pos_weight"
            ]
        ),

    "xgb-n-jobs":
        int(
            XGB_HYPERPARAMETERS[
                "n_jobs"
            ]
        ),
}


# ------------------------------------------------------------
# 4. Print exactly what will be sent
# ------------------------------------------------------------

print("\nTRAINING SCRIPT:")
print(staged_train)

print("\nTRAINING INPUT:")
print(
    "PreprocessHeartAttackData "
    "→ train.csv"
)

print("\nTRAINING ARGUMENTS:")
print("-" * 80)

for key, value in (
    training_hyperparameters.items()
):
    print(
        f"--{key:<25} {value}"
    )


# ------------------------------------------------------------
# 5. Define ModelTrainer
# ------------------------------------------------------------

trainer = ModelTrainer(

    training_image=
        SKLEARN_IMAGE_URI,

    source_code=
        training_source,

    compute=Compute(
        instance_type=
            training_instance_type,

        instance_count=1,
    ),

    role=
        ROLE_ARN,

    base_job_name=
        "team05-heart-xgboost-train",

    sagemaker_session=
        pipeline_session,

    hyperparameters=
        training_hyperparameters,

    input_data_config=[

        InputData(

            channel_name=
                "train",

            data_source=(
                step_process
                .properties
                .ProcessingOutputConfig
                .Outputs["train"]
                .S3Output
                .S3Uri
            ),

            content_type=
                "text/csv",
        )
    ],
)


# ------------------------------------------------------------
# 6. Convert trainer to pipeline TrainingStep
# ------------------------------------------------------------

train_args = trainer.train()


step_train = TrainingStep(

    name=
        "TrainHeartAttackXGBoost",

    step_args=
        train_args,

    cache_config=
        cache_config,
)


print("\n[OK] Training step defined.")

print(
    "✅ train.py receives --train-csv."
)

print(
    "✅ train.py receives --feature-metadata."
)

print(
    "✅ train.py receives --model xgboost."
)

print(
    "✅ Frozen XGBoost hyperparameters supplied."
)

SECTION 13 — TRAINING STEP

TRAINING SCRIPT:
/home/sagemaker-user/Heart_Attack_Risk_Assessment/artifacts/stage10_pipeline/source_bundle/train.py

TRAINING INPUT:
PreprocessHeartAttackData → train.csv

TRAINING ARGUMENTS:
--------------------------------------------------------------------------------
--train-csv                 /opt/ml/input/data/train/train.csv
--feature-metadata          /opt/ml/input/data/code/feature_metadata.json
--feature-set               full
--model                     xgboost
--output-dir                /opt/ml/model
--random-state              42
--xgb-n-estimators          150
--xgb-learning-rate         0.08
--xgb-max-depth             5
--xgb-subsample             0.75
--xgb-colsample-bytree      1.0
--xgb-reg-alpha             1.0
--xgb-reg-lambda            5.0
--xgb-scale-pos-weight      20.758675196654387
--xgb-n-jobs                2


[08/15/26 13:09:08] INFO     Cannot simulate policies for                                  ]8;id=8817015;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=8817016;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py#369\369]8;;\
                             'arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113                         
                             -Team05' (access denied); permission verdict unknown.                                 

                    WARNING  Could not verify permissions for role                         ]8;id=8817022;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=8817023;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py#591\591]8;;\
                             'arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113                         
                             -Team05' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'training' (see                                                       
                             IamRoleResolver().get_required_actions('training')) or create                         
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='training')                         
                             .                                                                                     

                    INFO     OutputDataConfig not provided. Using default:                          ]8;id=8817030;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=8817031;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#192\192]8;;\
                             s3_output_path='s3://sagemaker-ap-southeast-1-044528205969/team05-hear                
                             t-xgboost-train' kms_key_id=None compression_type='GZIP'                              

                    INFO     Training image URI:                                               ]8;id=8817038;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=8817039;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             121021644041.dkr.ecr.ap-southeast-1.amazonaws.com/sagemaker-sciki                     
                             t-learn:1.4-2-py312-cpu-py3                                                           


[OK] Training step defined.
✅ train.py receives --train-csv.
✅ train.py receives --feature-metadata.
✅ train.py receives --model xgboost.
✅ Frozen XGBoost hyperparameters supplied.


In [17]:
# ============================================================
# SECTION 13A — VALIDATE TRAINING STEP
# Does NOT start a training job
# ============================================================

print("=" * 80)
print("SECTION 13A — TRAINING STEP VALIDATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. Basic training step information
# ------------------------------------------------------------

print("\nTRAINING STEP")
print("-" * 80)

print("Step name :", step_train.name)
print("Step type :", type(step_train).__name__)


# ------------------------------------------------------------
# 2. Dependency on preprocessing
# ------------------------------------------------------------

print("\nDEPENDENCY CHECK")
print("-" * 80)

train_s3_expression = (
    step_process
    .properties
    .ProcessingOutputConfig
    .Outputs["train"]
    .S3Output
    .S3Uri
)

print("Upstream step:")
print(step_process.name)

print("\nTraining input comes from:")
print(train_s3_expression.expr)

print(
    "\n✅ Training data is linked to "
    "PreprocessHeartAttackData output."
)


# ------------------------------------------------------------
# 3. Input channels
# ------------------------------------------------------------

print("\nINPUT CHANNELS")
print("-" * 80)

input_configs = getattr(
    trainer,
    "input_data_config",
    []
)

if not input_configs:
    print("⚠️ No input_data_config found.")
else:
    for i, channel in enumerate(
        input_configs,
        start=1
    ):
        print(f"\nChannel {i}")

        # SDK objects can differ slightly between versions,
        # so use getattr safely.
        print(
            "Name        :",
            getattr(
                channel,
                "channel_name",
                "Unavailable"
            )
        )

        print(
            "Content type:",
            getattr(
                channel,
                "content_type",
                "Unavailable"
            )
        )

        data_source = getattr(
            channel,
            "data_source",
            None
        )

        if data_source is not None:

            if hasattr(
                data_source,
                "expr"
            ):
                print(
                    "Data source :",
                    data_source.expr
                )
            else:
                print(
                    "Data source :",
                    data_source
                )


# ------------------------------------------------------------
# 4. Model / XGBoost hyperparameters
# ------------------------------------------------------------

print("\nXGBOOST HYPERPARAMETERS")
print("-" * 80)

important_params = [
    "n_estimators",
    "max_depth",
    "learning_rate",
    "subsample",
    "colsample_bytree",
    "min_child_weight",
    "gamma",
    "reg_alpha",
    "reg_lambda",
    "scale_pos_weight",
    "objective",
    "eval_metric",
    "random_state",
    "n_jobs",
]

for name in important_params:

    if name in XGB_HYPERPARAMETERS:

        print(
            f"{name:<24}: "
            f"{XGB_HYPERPARAMETERS[name]}"
        )


# ------------------------------------------------------------
# 5. Training compute configuration
# ------------------------------------------------------------

print("\nTRAINING COMPUTE")
print("-" * 80)

print(
    "Instance type :",
    training_instance_type.expr
    if hasattr(
        training_instance_type,
        "expr"
    )
    else training_instance_type
)

print("Instance count: 1")


# ------------------------------------------------------------
# 6. Training script
# ------------------------------------------------------------

print("\nTRAINING SOURCE")
print("-" * 80)

print("Existing train.py:")
print(TRAIN_SCRIPT)

print("\nStaged train.py:")
print(staged_train)

print(
    "\nExists:",
    staged_train.exists()
)


# ------------------------------------------------------------
# 7. Clean-template check
# ------------------------------------------------------------

print("\nTRAINING TEMPLATE")
print("-" * 80)

print(
    "Template:",
    TEMPLATE_PIPELINE_FILE
)

print(
    "Exists  :",
    TEMPLATE_PIPELINE_FILE.exists()
)

print(
    "Legacy fitted state reused:",
    False
)


# ------------------------------------------------------------
# 8. Final validation
# ------------------------------------------------------------

assert (
    step_train.name
    == "TrainHeartAttackXGBoost"
)

assert (
    step_process.name
    == "PreprocessHeartAttackData"
)

assert staged_train.exists()

assert TEMPLATE_PIPELINE_FILE.exists()

assert len(
    XGB_HYPERPARAMETERS
) > 0


print("\n" + "=" * 80)
print("✅ SECTION 13 TRAINING STEP VALIDATION PASSED")
print("=" * 80)

print(
    "✅ Training input comes from preprocessing."
)

print(
    "✅ Existing train.py is staged."
)

print(
    "✅ XGBoost hyperparameters are available."
)

print(
    "✅ Clean unfitted template is available."
)

print(
    "\nSAFE TO CONTINUE TO SECTION 14."
)

SECTION 13A — TRAINING STEP VALIDATION

TRAINING STEP
--------------------------------------------------------------------------------
Step name : TrainHeartAttackXGBoost
Step type : TrainingStep

DEPENDENCY CHECK
--------------------------------------------------------------------------------
Upstream step:
PreprocessHeartAttackData

Training input comes from:
{'Get': "Steps.PreprocessHeartAttackData.ProcessingOutputConfig.Outputs['train'].S3Output.S3Uri"}

✅ Training data is linked to PreprocessHeartAttackData output.

INPUT CHANNELS
--------------------------------------------------------------------------------

Channel 1
Name        : train
Content type: text/csv
Data source : {'Get': "Steps.PreprocessHeartAttackData.ProcessingOutputConfig.Outputs['train'].S3Output.S3Uri"}

XGBOOST HYPERPARAMETERS
--------------------------------------------------------------------------------
n_estimators            : 150
max_depth               : 5
learning_rate           : 0.08
subsample       

## 14. FIRST RUN — Evaluate trained model using EXISTING `evaluate.py`

The SageMaker training job produces `model.tar.gz`. This section uses a small pipeline adapter to extract `model.joblib`, pass the validation data and `feature_metadata.json`, call the existing `evaluate.py`, and create `evaluation.json` for the quality gate and Model Registry.

The existing `src/evaluate.py` evaluation logic is reused rather than replaced.


In [18]:
# ============================================================
# SECTION 14A — PREPARE EVALUATION SUPPORT FILES
# ============================================================

from pathlib import Path
import shutil
import boto3


print("=" * 80)
print("SECTION 14A — PREPARE EVALUATION SUPPORT FILES")
print("=" * 80)


# ------------------------------------------------------------
# Existing evaluator
# ------------------------------------------------------------

EXISTING_EVALUATE_SCRIPT = (
    PROJECT_ROOT
    / "src"
    / "evaluate.py"
)

if not EXISTING_EVALUATE_SCRIPT.exists():
    raise FileNotFoundError(
        f"evaluate.py not found: "
        f"{EXISTING_EVALUATE_SCRIPT}"
    )


# ------------------------------------------------------------
# Existing feature metadata
# ------------------------------------------------------------

FEATURE_METADATA_SOURCE = (
    PROJECT_ROOT
    / "config"
    / "feature_metadata.json"
)

if not FEATURE_METADATA_SOURCE.exists():
    raise FileNotFoundError(
        f"feature_metadata.json not found: "
        f"{FEATURE_METADATA_SOURCE}"
    )


# ------------------------------------------------------------
# Upload support files to S3
# ------------------------------------------------------------

EVALUATE_SCRIPT_S3_KEY = (
    f"{S3_PREFIX}/evaluation-support/"
    "evaluate.py"
)

FEATURE_METADATA_S3_KEY = (
    f"{S3_PREFIX}/evaluation-support/"
    "feature_metadata.json"
)


EVALUATE_SCRIPT_S3_URI = (
    f"s3://{BUCKET}/"
    f"{EVALUATE_SCRIPT_S3_KEY}"
)

FEATURE_METADATA_S3_URI = (
    f"s3://{BUCKET}/"
    f"{FEATURE_METADATA_S3_KEY}"
)


s3_client.upload_file(
    str(EXISTING_EVALUATE_SCRIPT),
    BUCKET,
    EVALUATE_SCRIPT_S3_KEY,
)

s3_client.upload_file(
    str(FEATURE_METADATA_SOURCE),
    BUCKET,
    FEATURE_METADATA_S3_KEY,
)


print("\nExisting evaluator:")
print(EXISTING_EVALUATE_SCRIPT)

print("\nEvaluator S3:")
print(EVALUATE_SCRIPT_S3_URI)

print("\nFeature metadata:")
print(FEATURE_METADATA_SOURCE)

print("\nMetadata S3:")
print(FEATURE_METADATA_S3_URI)


print("\n✅ Existing evaluate.py uploaded.")
print("✅ feature_metadata.json uploaded.")

SECTION 14A — PREPARE EVALUATION SUPPORT FILES

Existing evaluator:
/home/sagemaker-user/Heart_Attack_Risk_Assessment/src/evaluate.py

Evaluator S3:
s3://sagemaker-ap-southeast-1-044528205969/heart-attack-risk/pipeline10/evaluation-support/evaluate.py

Feature metadata:
/home/sagemaker-user/Heart_Attack_Risk_Assessment/config/feature_metadata.json

Metadata S3:
s3://sagemaker-ap-southeast-1-044528205969/heart-attack-risk/pipeline10/evaluation-support/feature_metadata.json

✅ Existing evaluate.py uploaded.
✅ feature_metadata.json uploaded.


In [19]:
# ============================================================
# SECTION 14B — CREATE EVALUATION ADAPTER
#
# The adapter does NOT replace evaluate.py.
#
# It only:
#   1. extracts model.tar.gz
#   2. locates model.joblib
#   3. calls existing evaluate.py
# ============================================================

EVALUATION_ADAPTER_SCRIPT = (
    PIPELINE_WORK_DIR
    / "evaluate_pipeline_adapter.py"
)


adapter_code = r'''
import sys
import tarfile
import subprocess
from pathlib import Path


MODEL_DIR = Path(
    "/opt/ml/processing/model"
)

VALIDATION_DIR = Path(
    "/opt/ml/processing/validation"
)

EVALUATOR_DIR = Path(
    "/opt/ml/processing/evaluator"
)

METADATA_DIR = Path(
    "/opt/ml/processing/metadata"
)

OUTPUT_DIR = Path(
    "/opt/ml/processing/evaluation"
)

EXTRACT_DIR = (
    MODEL_DIR / "extracted"
)

THRESHOLD = "0.52"

FEATURE_SET = "full"


print("=" * 80)
print("PIPELINE EVALUATION ADAPTER")
print("=" * 80)


# ------------------------------------------------------------
# Install XGBoost if the processing image does not contain it
# ------------------------------------------------------------

try:
    import xgboost

    print(
        "XGBoost available:",
        xgboost.__version__
    )

except ImportError:

    print(
        "XGBoost not found. "
        "Installing xgboost==3.4.1..."
    )

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "xgboost==3.4.1",
        ]
    )


# ------------------------------------------------------------
# Locate validation.csv
# ------------------------------------------------------------

validation_files = list(
    VALIDATION_DIR.rglob(
        "validation.csv"
    )
)

if not validation_files:
    raise FileNotFoundError(
        "validation.csv not found under "
        "/opt/ml/processing/validation"
    )

validation_csv = (
    validation_files[0]
)

print(
    "\nValidation CSV:",
    validation_csv
)


# ------------------------------------------------------------
# Locate model artifact
# ------------------------------------------------------------

model_joblib_files = list(
    MODEL_DIR.rglob(
        "model.joblib"
    )
)


if model_joblib_files:

    model_joblib = (
        model_joblib_files[0]
    )

else:

    archives = list(
        MODEL_DIR.rglob(
            "*.tar.gz"
        )
    )

    if not archives:
        raise FileNotFoundError(
            "Neither model.joblib nor "
            "model.tar.gz was found."
        )

    archive = archives[0]

    print(
        "\nModel archive:",
        archive
    )

    EXTRACT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    with tarfile.open(
        archive,
        "r:gz"
    ) as tar:

        tar.extractall(
            EXTRACT_DIR
        )

    extracted_models = list(
        EXTRACT_DIR.rglob(
            "model.joblib"
        )
    )

    if not extracted_models:
        raise FileNotFoundError(
            "model.joblib was not found "
            "after extracting model.tar.gz."
        )

    model_joblib = (
        extracted_models[0]
    )


print(
    "Model joblib:",
    model_joblib
)


# ------------------------------------------------------------
# Locate existing evaluate.py
# ------------------------------------------------------------

evaluate_files = list(
    EVALUATOR_DIR.rglob(
        "evaluate.py"
    )
)

if not evaluate_files:
    raise FileNotFoundError(
        "Existing evaluate.py not found."
    )

evaluate_script = (
    evaluate_files[0]
)

print(
    "Existing evaluator:",
    evaluate_script
)


# ------------------------------------------------------------
# Locate feature metadata
# ------------------------------------------------------------

metadata_files = list(
    METADATA_DIR.rglob(
        "feature_metadata.json"
    )
)

if not metadata_files:
    raise FileNotFoundError(
        "feature_metadata.json not found."
    )

metadata_file = (
    metadata_files[0]
)

print(
    "Feature metadata:",
    metadata_file
)


# ------------------------------------------------------------
# Create evaluation output directory
# ------------------------------------------------------------

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Call EXISTING evaluate.py
# ------------------------------------------------------------

command = [
    sys.executable,

    str(
        evaluate_script
    ),

    "--test-csv",
    str(
        validation_csv
    ),

    "--model-path",
    str(
        model_joblib
    ),

    "--feature-metadata",
    str(
        metadata_file
    ),

    "--feature-set",
    FEATURE_SET,

    "--threshold",
    THRESHOLD,

    "--output-dir",
    str(
        OUTPUT_DIR
    ),
]


print("\nCalling existing evaluate.py:")
print(" ".join(command))


result = subprocess.run(
    command,
    text=True,
)


if result.returncode != 0:

    raise RuntimeError(
        "Existing evaluate.py failed "
        f"with exit code "
        f"{result.returncode}."
    )


evaluation_json = (
    OUTPUT_DIR
    / "evaluation.json"
)


if not evaluation_json.exists():

    raise FileNotFoundError(
        "evaluate.py completed but "
        "evaluation.json was not created."
    )


print(
    "\n✅ Existing evaluate.py completed."
)

print(
    "✅ evaluation.json created:"
)

print(
    evaluation_json
)
'''


EVALUATION_ADAPTER_SCRIPT.write_text(
    adapter_code,
    encoding="utf-8",
)


print(
    "Adapter created:"
)

print(
    EVALUATION_ADAPTER_SCRIPT
)

print(
    "\nExists:",
    EVALUATION_ADAPTER_SCRIPT.exists()
)

print(
    "\n✅ Evaluation adapter ready."
)

Adapter created:
/home/sagemaker-user/Heart_Attack_Risk_Assessment/artifacts/stage10_pipeline/evaluate_pipeline_adapter.py

Exists: True

✅ Evaluation adapter ready.


In [20]:
# ============================================================
# SECTION 14C — DEFINE EVALUATION PROCESSING STEP
# ============================================================

from sagemaker.core.processing import (
    ScriptProcessor,
)

from sagemaker.core.shapes import (
    ProcessingInput,
    ProcessingS3Input,
    ProcessingOutput,
    ProcessingS3Output,
)

from sagemaker.mlops.workflow.steps import (
    ProcessingStep,
)

from sagemaker.core.workflow.properties import (
    PropertyFile,
)


print("=" * 80)
print("SECTION 14C — DEFINE EVALUATION STEP")
print("=" * 80)


# ------------------------------------------------------------
# Property file
# ------------------------------------------------------------

evaluation_report = PropertyFile(
    name="HeartAttackEvaluationReport",
    output_name="evaluation",
    path="evaluation.json",
)


# ------------------------------------------------------------
# Evaluation processor
# ------------------------------------------------------------

eval_processor = ScriptProcessor(

    image_uri=
        SKLEARN_IMAGE_URI,

    role=
        ROLE_ARN,

    instance_count=
        1,

    instance_type=
        processing_instance_type,

    command=[
        "python3"
    ],

    base_job_name=
        "team05-heart-evaluate",

    sagemaker_session=
        pipeline_session,
)


# ------------------------------------------------------------
# Run configuration
# ------------------------------------------------------------

eval_args = eval_processor.run(

    code=str(
        EVALUATION_ADAPTER_SCRIPT
    ),

    inputs=[

        # ----------------------------------------------------
        # MODEL from training
        # ----------------------------------------------------

        ProcessingInput(

            input_name=
                "model",

            s3_input=
                ProcessingS3Input(

                    s3_uri=(
                        step_train
                        .properties
                        .ModelArtifacts
                        .S3ModelArtifacts
                    ),

                    local_path=
                        "/opt/ml/processing/model",

                    s3_data_type=
                        "S3Prefix",

                    s3_input_mode=
                        "File",

                    s3_data_distribution_type=
                        "FullyReplicated",
                ),
        ),


        # ----------------------------------------------------
        # VALIDATION data from preprocessing
        # ----------------------------------------------------

        ProcessingInput(

            input_name=
                "validation",

            s3_input=
                ProcessingS3Input(

                    s3_uri=(
                        step_process
                        .properties
                        .ProcessingOutputConfig
                        .Outputs["validation"]
                        .S3Output
                        .S3Uri
                    ),

                    local_path=
                        "/opt/ml/processing/validation",

                    s3_data_type=
                        "S3Prefix",

                    s3_input_mode=
                        "File",

                    s3_data_distribution_type=
                        "FullyReplicated",
                ),
        ),


        # ----------------------------------------------------
        # EXISTING evaluate.py
        # ----------------------------------------------------

        ProcessingInput(

            input_name=
                "evaluator",

            s3_input=
                ProcessingS3Input(

                    s3_uri=
                        EVALUATE_SCRIPT_S3_URI,

                    local_path=
                        "/opt/ml/processing/evaluator",

                    s3_data_type=
                        "S3Prefix",

                    s3_input_mode=
                        "File",

                    s3_data_distribution_type=
                        "FullyReplicated",
                ),
        ),


        # ----------------------------------------------------
        # FEATURE METADATA
        # ----------------------------------------------------

        ProcessingInput(

            input_name=
                "feature-metadata",

            s3_input=
                ProcessingS3Input(

                    s3_uri=
                        FEATURE_METADATA_S3_URI,

                    local_path=
                        "/opt/ml/processing/metadata",

                    s3_data_type=
                        "S3Prefix",

                    s3_input_mode=
                        "File",

                    s3_data_distribution_type=
                        "FullyReplicated",
                ),
        ),
    ],


    # --------------------------------------------------------
    # Evaluation output
    # --------------------------------------------------------

    outputs=[

        ProcessingOutput(

            output_name=
                "evaluation",

            s3_output=
                ProcessingS3Output(

                    s3_uri=(
                        f"s3://{BUCKET}/"
                        f"{S3_PREFIX}/evaluation"
                    ),

                    local_path=
                        "/opt/ml/processing/evaluation",

                    s3_upload_mode=
                        "EndOfJob",
                ),
        )
    ],
)


# ------------------------------------------------------------
# Pipeline step
# ------------------------------------------------------------

step_evaluate = ProcessingStep(

    name=
        "EvaluateHeartAttackModel",

    step_args=
        eval_args,

    property_files=[
        evaluation_report
    ],

    cache_config=
        cache_config,
)


print(
    "\n[OK] Evaluation step defined."
)

print(
    "✅ Model artifact comes from training."
)

print(
    "✅ Validation data comes from preprocessing."
)

print(
    "✅ Existing evaluate.py is supplied."
)

print(
    "✅ feature_metadata.json is supplied."
)

print(
    "✅ model.tar.gz will be extracted automatically by adapter."
)

print(
    "✅ Frozen threshold =",
    FROZEN_THRESHOLD
)

SECTION 14C — DEFINE EVALUATION STEP

[OK] Evaluation step defined.
✅ Model artifact comes from training.
✅ Validation data comes from preprocessing.
✅ Existing evaluate.py is supplied.
✅ feature_metadata.json is supplied.
✅ model.tar.gz will be extracted automatically by adapter.
✅ Frozen threshold = 0.52


In [21]:
# ============================================================
# SECTION 14D — VALIDATE EVALUATION STEP
# ============================================================

print("=" * 80)
print("SECTION 14D — EVALUATION VALIDATION")
print("=" * 80)


print("\nEvaluation step:")
print(step_evaluate.name)


print("\nModel dependency:")
print(
    step_train
    .properties
    .ModelArtifacts
    .S3ModelArtifacts
    .expr
)


print("\nValidation dependency:")
print(
    step_process
    .properties
    .ProcessingOutputConfig
    .Outputs["validation"]
    .S3Output
    .S3Uri
    .expr
)


print("\nExisting evaluator:")
print(EXISTING_EVALUATE_SCRIPT)


print("\nFeature metadata:")
print(FEATURE_METADATA_SOURCE)


print("\nAdapter:")
print(EVALUATION_ADAPTER_SCRIPT)


assert (
    step_evaluate.name
    == "EvaluateHeartAttackModel"
)

assert (
    EVALUATION_ADAPTER_SCRIPT.exists()
)

assert (
    EXISTING_EVALUATE_SCRIPT.exists()
)

assert (
    FEATURE_METADATA_SOURCE.exists()
)

assert (
    evaluation_report.path
    == "evaluation.json"
)


print("\n" + "=" * 80)
print("✅ SECTION 14 VALIDATION PASSED")
print("=" * 80)

print(
    "SAFE TO CONTINUE TO SECTION 15."
)

SECTION 14D — EVALUATION VALIDATION

Evaluation step:
EvaluateHeartAttackModel

Model dependency:
{'Get': 'Steps.TrainHeartAttackXGBoost.ModelArtifacts.S3ModelArtifacts'}

Validation dependency:
{'Get': "Steps.PreprocessHeartAttackData.ProcessingOutputConfig.Outputs['validation'].S3Output.S3Uri"}

Existing evaluator:
/home/sagemaker-user/Heart_Attack_Risk_Assessment/src/evaluate.py

Feature metadata:
/home/sagemaker-user/Heart_Attack_Risk_Assessment/config/feature_metadata.json

Adapter:
/home/sagemaker-user/Heart_Attack_Risk_Assessment/artifacts/stage10_pipeline/evaluate_pipeline_adapter.py

✅ SECTION 14 VALIDATION PASSED
SAFE TO CONTINUE TO SECTION 15.


## 15. FIRST RUN — Model Registry step


In [22]:
# ============================================================
# SECTION 15 — MODEL REGISTRY STEP
# ============================================================

from sagemaker.core.workflow.functions import (
    Join,
)

from sagemaker.core.model_metrics import (
    MetricsSource,
    ModelMetrics,
)

from sagemaker.serve.model_builder import (
    ModelBuilder,
)

from sagemaker.mlops.workflow.model_step import (
    ModelStep,
)


print("=" * 80)
print("SECTION 15 — MODEL REGISTRY")
print("=" * 80)


# ------------------------------------------------------------
# Evaluation output S3 expression
# ------------------------------------------------------------

evaluation_s3_uri = (
    step_evaluate
    .properties
    .ProcessingOutputConfig
    .Outputs["evaluation"]
    .S3Output
    .S3Uri
)


evaluation_json_uri = Join(

    on="/",

    values=[
        evaluation_s3_uri,
        "evaluation.json",
    ],
)


print(
    "Evaluation JSON expression:"
)

print(
    evaluation_json_uri.expr
)


# ------------------------------------------------------------
# Registry metrics
# ------------------------------------------------------------

model_metrics = ModelMetrics(

    model_statistics=MetricsSource(

        s3_uri=
            evaluation_json_uri,

        content_type=
            "application/json",
    )
)


# ------------------------------------------------------------
# Registered model
# ------------------------------------------------------------

registry_builder = ModelBuilder(

    image_uri=
        SKLEARN_IMAGE_URI,

    s3_model_data_url=(
        step_train
        .properties
        .ModelArtifacts
        .S3ModelArtifacts
    ),

    role_arn=
        ROLE_ARN,

    sagemaker_session=
        pipeline_session,
)


register_args = (
    registry_builder.register(

        model_package_group_name=
            MODEL_PACKAGE_GROUP,

        content_types=[
            "application/json"
        ],

        response_types=[
            "application/json"
        ],

        inference_instances=[
            "ml.m5.large"
        ],

        approval_status=
            model_approval_status,

        model_metrics=
            model_metrics,
    )
)


step_register = ModelStep(

    name=
        "RegisterHeartAttackModel",

    step_args=
        register_args,
)


print(
    "\n✅ Model Registry step defined."
)

print(
    "Package group:",
    MODEL_PACKAGE_GROUP
)

print(
    "Approval:",
    model_approval_status.expr
)

SECTION 15 — MODEL REGISTRY
Evaluation JSON expression:
{'Std:Join': {'On': '/', 'Values': [{'Get': "Steps.EvaluateHeartAttackModel.ProcessingOutputConfig.Outputs['evaluation'].S3Output.S3Uri"}, 'evaluation.json']}}


[08/15/26 13:09:33] DEBUG    Auto-detecting optimal instance type for model...           ]8;id=8817046;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=8817047;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#341\341]8;;\

                    DEBUG    Using default CPU instance type: ml.m5.large                ]8;id=8817053;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=8817054;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#375\375]8;;\


✅ Model Registry step defined.
Package group: iti113-team05-heart-attack-risk-models
Approval: {'Get': 'Parameters.ModelApprovalStatus'}


## 16. FIRST RUN — PR-AUC + Recall quality gate

The existing `evaluate.py` writes flat `recall` and `pr_auc` values to `evaluation.json`; the quality gate reads those two top-level JSON paths.


In [23]:
# ============================================================
# SECTION 16 — QUALITY GATE
# ============================================================

from sagemaker.core.workflow.functions import (
    JsonGet,
)

from sagemaker.core.workflow.conditions import (
    ConditionGreaterThanOrEqualTo,
)

from sagemaker.mlops.workflow.condition_step import (
    ConditionStep,
)


print("=" * 80)
print("SECTION 16 — QUALITY GATE")
print("=" * 80)


# ------------------------------------------------------------
# Existing evaluate.py writes FLAT JSON:
#
# {
#     "recall": ...,
#     "pr_auc": ...
# }
# ------------------------------------------------------------

pr_auc_value = JsonGet(

    step_name=
        step_evaluate.name,

    property_file=
        evaluation_report,

    json_path=
        "pr_auc",
)


recall_value = JsonGet(

    step_name=
        step_evaluate.name,

    property_file=
        evaluation_report,

    json_path=
        "recall",
)


# ------------------------------------------------------------
# Gate
# ------------------------------------------------------------

step_quality_gate = ConditionStep(

    name=
        "HeartAttackQualityGate",

    conditions=[

        ConditionGreaterThanOrEqualTo(

            left=
                pr_auc_value,

            right=
                min_pr_auc,
        ),

        ConditionGreaterThanOrEqualTo(

            left=
                recall_value,

            right=
                min_recall,
        ),
    ],

    if_steps=[
        step_register
    ],

    else_steps=[],
)


print(
    "Minimum PR-AUC:",
    min_pr_auc.expr
)

print(
    "Minimum Recall:",
    min_recall.expr
)

print(
    "\n✅ Quality gate defined."
)

print(
    "✅ JSON path = pr_auc"
)

print(
    "✅ JSON path = recall"
)

SECTION 16 — QUALITY GATE
Minimum PR-AUC: {'Get': 'Parameters.MinimumPRAUC'}
Minimum Recall: {'Get': 'Parameters.MinimumRecall'}

✅ Quality gate defined.
✅ JSON path = pr_auc
✅ JSON path = recall


## 17. FIRST RUN — Assemble pipeline

In [24]:
# ============================================================
# SECTION 17 — ASSEMBLE PIPELINE
# ============================================================

import json


print("=" * 80)
print("SECTION 17 — ASSEMBLE PIPELINE")
print("=" * 80)


pipeline = SageMakerPipeline(

    name=
        PIPELINE_NAME,

    parameters=[

        processing_instance_count,

        processing_instance_type,

        training_instance_type,

        input_data,

        model_approval_status,

        min_pr_auc,

        min_recall,
    ],

    steps=[

        step_process,

        step_train,

        step_evaluate,

        step_quality_gate,
    ],

    sagemaker_session=
        pipeline_session,
)


definition = (
    pipeline.definition()
)


# Verify valid JSON
json.loads(
    definition
)


print(
    "Pipeline:",
    PIPELINE_NAME
)

print(
    "\n✅ Pipeline definition generated."
)

print(
    "✅ Pipeline definition is valid JSON."
)

SECTION 17 — ASSEMBLE PIPELINE


[08/15/26 13:09:44] WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=8817061;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=8817062;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

[08/15/26 13:09:45] DEBUG    Could not verify role access to default bucket: An error occurred ]8;id=8817068;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=8817069;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#911\911]8;;\
                             (AccessDenied) when calling the SimulatePrincipalPolicy                               
                             operation: User:                                                                      
                             arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI                     
                             113-Team05/SageMaker is not authorized to perform:                                    
                             iam:SimulatePrincipalPolicy on resource:                                              
                             arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team                     
                             05 because no identity-based policy allows the                                        
                             iam:SimulatePrincipalPolicy action                                                    

[08/15/26 13:09:46] DEBUG    Could not verify role access to default bucket: An error occurred ]8;id=8817074;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=8817075;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#911\911]8;;\
                             (AccessDenied) when calling the SimulatePrincipalPolicy                               
                             operation: User:                                                                      
                             arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI                     
                             113-Team05/SageMaker is not authorized to perform:                                    
                             iam:SimulatePrincipalPolicy on resource:                                              
                             arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team                     
                             05 because no identity-based policy allows the                                        
                             iam:SimulatePrincipalPolicy action                                                    

[08/15/26 13:09:47] WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=8817080;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=8817081;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=8817086;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=8817087;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'CertifyForMarketplace' from the pipeline definition     ]8;id=8817094;file:///opt/conda/lib/python3.12/site-packages/sagemaker/mlops/workflow/model_step.py\model_step.py]8;;\:]8;id=8817095;file:///opt/conda/lib/python3.12/site-packages/sagemaker/mlops/workflow/model_step.py#195\195]8;;\
                             since it will be overridden in pipeline execution time.                               

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=8817100;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=8817101;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

Pipeline: iti113-team05-heart-attack-risk

✅ Pipeline definition generated.
✅ Pipeline definition is valid JSON.


## 17A. FINAL PRE-UPSERT VALIDATION


In [25]:
# ============================================================
# SECTION 17A — FINAL PRE-UPSERT VALIDATION
# ============================================================

print("=" * 80)
print("FINAL PIPELINE VALIDATION")
print("=" * 80)


definition = (
    pipeline.definition()
)


checks = {

    "full_train_raw.csv":
        "full_train_raw.csv"
        in definition,

    "--validation-size":
        "--validation-size"
        in definition,

    "TrainHeartAttackXGBoost":
        "TrainHeartAttackXGBoost"
        in definition,

    "EvaluateHeartAttackModel":
        "EvaluateHeartAttackModel"
        in definition,

    "feature-metadata":
        "feature-metadata"
        in definition,

    "evaluation":
        "evaluation"
        in definition,

    "HeartAttackQualityGate":
        "HeartAttackQualityGate"
        in definition,

    "RegisterHeartAttackModel":
        "RegisterHeartAttackModel"
        in definition,
}


print()

for name, passed in checks.items():

    print(
        f"{name:<32}: "
        f"{'✅ FOUND' if passed else '❌ MISSING'}"
    )


if not all(
    checks.values()
):

    raise RuntimeError(
        "\nPipeline validation failed. "
        "DO NOT UPSERT."
    )


print("\n" + "=" * 80)

print(
    "✅ FINAL PIPELINE VALIDATION PASSED"
)

print("=" * 80)

print(
    "\nSAFE TO UPSERT PIPELINE."
)

FINAL PIPELINE VALIDATION


[08/15/26 13:09:51] WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=8817106;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=8817107;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

[08/15/26 13:09:52] DEBUG    Could not verify role access to default bucket: An error occurred ]8;id=8817112;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=8817113;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#911\911]8;;\
                             (AccessDenied) when calling the SimulatePrincipalPolicy                               
                             operation: User:                                                                      
                             arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI                     
                             113-Team05/SageMaker is not authorized to perform:                                    
                             iam:SimulatePrincipalPolicy on resource:                                              
                             arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team                     
                             05 because no identity-based policy allows the                                        
                             iam:SimulatePrincipalPolicy action                                                    

[08/15/26 13:09:53] DEBUG    Could not verify role access to default bucket: An error occurred ]8;id=8817118;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=8817119;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#911\911]8;;\
                             (AccessDenied) when calling the SimulatePrincipalPolicy                               
                             operation: User:                                                                      
                             arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI                     
                             113-Team05/SageMaker is not authorized to perform:                                    
                             iam:SimulatePrincipalPolicy on resource:                                              
                             arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team                     
                             05 because no identity-based policy allows the                                        
                             iam:SimulatePrincipalPolicy action                                                    

                    WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=8817124;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=8817125;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=8817130;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=8817131;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=8817136;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=8817137;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   


full_train_raw.csv              : ✅ FOUND
--validation-size               : ✅ FOUND
TrainHeartAttackXGBoost         : ✅ FOUND
EvaluateHeartAttackModel        : ✅ FOUND
feature-metadata                : ✅ FOUND
evaluation                      : ✅ FOUND
HeartAttackQualityGate          : ✅ FOUND
RegisterHeartAttackModel        : ✅ FOUND

✅ FINAL PIPELINE VALIDATION PASSED

SAFE TO UPSERT PIPELINE.


## 18. FIRST RUN — Create/update pipeline in SageMaker

After this succeeds, `iti113-team05-heart-attack-risk` should appear under **SageMaker → Pipelines**.

In [26]:
# ============================================================
# SECTION 18 — UPSERT PIPELINE
# ============================================================

import json


print("=" * 80)
print("SECTION 18 — UPSERT PIPELINE")
print("=" * 80)


upsert_response = (
    pipeline.upsert(
        role_arn=
            ROLE_ARN
    )
)


print(
    json.dumps(
        upsert_response,
        indent=2,
        default=str,
    )
)


print(
    "\n✅ Pipeline upserted:"
)

print(
    PIPELINE_NAME
)

SECTION 18 — UPSERT PIPELINE


[08/15/26 13:10:05] WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=8817142;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=8817143;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    DEBUG    Could not verify role access to default bucket: An error occurred ]8;id=8817148;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=8817149;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#911\911]8;;\
                             (AccessDenied) when calling the SimulatePrincipalPolicy                               
                             operation: User:                                                                      
                             arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI                     
                             113-Team05/SageMaker is not authorized to perform:                                    
                             iam:SimulatePrincipalPolicy on resource:                                              
                             arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team                     
                             05 because no identity-based policy allows the                                        
                             iam:SimulatePrincipalPolicy action                                                    

[08/15/26 13:10:06] DEBUG    Could not verify role access to default bucket: An error occurred ]8;id=8817154;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=8817155;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#911\911]8;;\
                             (AccessDenied) when calling the SimulatePrincipalPolicy                               
                             operation: User:                                                                      
                             arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI                     
                             113-Team05/SageMaker is not authorized to perform:                                    
                             iam:SimulatePrincipalPolicy on resource:                                              
                             arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team                     
                             05 because no identity-based policy allows the                                        
                             iam:SimulatePrincipalPolicy action                                                    

[08/15/26 13:10:07] WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=8817160;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=8817161;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

...

[08/15/26 13:10:10] DEBUG    Could not verify role access to default bucket: An error occurred ]8;id=8817190;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=8817191;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#911\911]8;;\
                             (AccessDenied) when calling the SimulatePrincipalPolicy                               
                             operation: User:                                                                      
                             arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI                     
                             113-Team05/SageMaker is not authorized to perform:                                    
                             iam:SimulatePrincipalPolicy on resource:                                              
                             arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team                     
                             05 because no identity-based policy allows the                                        
                             iam:SimulatePrincipalPolicy action                                                    

[08/15/26 13:10:11] WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=8817196;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=8817197;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=8817202;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=8817203;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=8817208;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=8817209;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

{
  "PipelineArn": "arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/iti113-team05-heart-attack-risk",
  "PipelineVersionId": 7,
  "ResponseMetadata": {
    "RequestId": "35e9c59f-e57c-4e70-85a4-b60361f25fed",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "x-amzn-requestid": "35e9c59f-e57c-4e70-85a4-b60361f25fed",
      "strict-transport-security": "max-age=47304000; includeSubDomains",
      "x-frame-options": "DENY",
      "content-security-policy": "frame-ancestors 'none'",
      "cache-control": "no-cache, no-store, must-revalidate",
      "x-content-type-options": "nosniff",
      "content-type": "application/x-amz-json-1.1",
      "content-length": "126",
      "date": "Sat, 15 Aug 2026 13:10:11 GMT"
    },
    "RetryAttempts": 0
  }
}

✅ Pipeline upserted:
iti113-team05-heart-attack-risk


## 19. NEW MODEL VERSION — Start ONE pipeline execution

Run this cell **once only**.

Every time this cell is executed, SageMaker starts another pipeline execution.

In [27]:
# ============================================================
# SECTION 19 — START ONE PIPELINE EXECUTION
# ============================================================

from datetime import (
    datetime,
    timezone,
)


print("=" * 80)
print("SECTION 19 — START PIPELINE EXECUTION")
print("=" * 80)


execution_start = (
    datetime.now(
        timezone.utc
    )
)


execution = pipeline.start(

    parameters={

        "ModelApprovalStatus":
            "PendingManualApproval",

        "MinimumPRAUC":
            0.35,

        "MinimumRecall":
            0.75,
    }
)


print(
    "\n✅ NEW execution started."
)

print(
    "\nStarted:"
)

print(
    execution_start
)

print(
    "\nExecution ARN:"
)

print(
    execution.arn
)

print(
    "\nIMPORTANT:"
)

print(
    "Do NOT run this cell again "
    "while this execution is running."
)

SECTION 19 — START PIPELINE EXECUTION



✅ NEW execution started.

Started:
2026-08-15 13:10:15.325722+00:00

Execution ARN:
arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/iti113-team05-heart-attack-risk/execution/xldnxhku9k3r

IMPORTANT:
Do NOT run this cell again while this execution is running.


## 20. OPTIONAL — Check current execution status

This cell is safe to run repeatedly. It does **not** start another pipeline execution.


In [28]:
# ============================================================
# SECTION 20 — CHECK CURRENT EXECUTION
# SAFE TO RUN REPEATEDLY
# ============================================================

print("=" * 80)
print("CURRENT PIPELINE EXECUTION")
print("=" * 80)


print(
    "Execution ARN:"
)

print(
    execution.arn
)


desc = (
    execution.describe()
)


status = (
    desc.get(
        "PipelineExecutionStatus"
    )
)


print(
    "\nPipeline status:",
    status
)


print("\nSteps:")
print("-" * 80)


steps = (
    execution.list_steps()
)


for step in reversed(steps):

    print(
        f"{step.get('StepName', ''):<32}"
        f"{step.get('StepStatus', '')}"
    )

    if step.get(
        "FailureReason"
    ):

        print(
            "   Failure:",
            step.get(
                "FailureReason"
            )
        )


print("\n" + "=" * 80)


if status == "Succeeded":

    print(
        "✅ PIPELINE EXECUTION SUCCEEDED"
    )

elif status == "Failed":

    print(
        "❌ PIPELINE EXECUTION FAILED"
    )

    print(
        "Inspect only the failed step."
    )

else:

    print(
        "⏳ Pipeline is still running."
    )

CURRENT PIPELINE EXECUTION
Execution ARN:
arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/iti113-team05-heart-attack-risk/execution/xldnxhku9k3r

Pipeline status: Succeeded

Steps:
--------------------------------------------------------------------------------


PreprocessHeartAttackData       Succeeded
TrainHeartAttackXGBoost         Succeeded
EvaluateHeartAttackModel        Succeeded
HeartAttackQualityGate          Succeeded
RegisterHeartAttackModel        Succeeded

✅ PIPELINE EXECUTION SUCCEEDED


## 14   Evaluation
 │
 ├─14A Upload existing evaluate.py + feature_metadata.json
 ├─14B Create evaluation adapter
 ├─14C Define evaluation ProcessingStep
 └─14D Validate evaluation
       ↓
15   Model Registry
       ↓
16   PR-AUC + Recall Quality Gate
       ↓
17   Assemble Pipeline
       ↓
17A  Final Pre-Upsert Validation
       ↓
18   Upsert Pipeline
       ↓
19   Start ONE Execution
       ↓
20   Check Execution Status

# Final order

```text
01–09 Existing ML / Responsible AI / holdout / governance
  ↓
10_SageMaker_MLOps_Pipeline.ipynb
  Existing preprocess.py → existing train.py → existing evaluate.py
  → quality gate → model registry
  ↓
11_SageMaker_Deployment_Endpoint.ipynb
  ↓
12_Gradio_System_Demo.ipynb
```

### Important
This notebook deliberately **reuses your existing source code** instead of maintaining a second preprocessing/training/evaluation implementation.

If Section 6 finds multiple versions of a script, set the explicit override path in Section 6 before proceeding.